# 07: Hyperparameter tuning on the calibrated pipeline

Notebook 06 established a trustworthy setup: CV 0.94925 vs LB 0.94973 under
balanced accuracy. This notebook asks whether tuning HistGradientBoosting's
hyperparameters improves on the defaults. Randomized search, 30 candidates,
3-fold stratified CV, scored on balanced accuracy. Baseline to beat: 0.94925.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import HistGradientBoostingClassifier

train = pd.read_csv('../data/raw/train.csv')
test = pd.read_csv('../data/raw/test.csv')

X = train.drop(columns=['id', 'health_condition'])
y = train['health_condition']
X_test = test.drop(columns=['id'])

def add_features(df):
    df = df.copy()
    df['stress_activity_combo'] = (df['stress_level'].astype(str) + '_' +
                                   df['physical_activity_level'].astype(str))
    df['activity_score'] = df['step_count'] * df['exercise_duration']
    df['sleep_activity_ratio'] = df['sleep_duration'] / (df['exercise_duration'] + 1)
    return df.drop(columns=['diet_type', 'gender'])

X = add_features(X)
X_test = add_features(X_test)

cat_cols = ['stress_level', 'sleep_quality', 'physical_activity_level',
            'smoking_alcohol', 'stress_activity_combo']
num_cols = [c for c in X.columns if c not in cat_cols]

encode = ColumnTransformer([
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=np.nan,
                           encoded_missing_value=np.nan), cat_cols),
    ('num', 'passthrough', num_cols),
])

In [2]:
pipe = Pipeline([
    ('encode', encode),
    ('clf', HistGradientBoostingClassifier(class_weight='balanced', random_state=42)),
])

param_distributions = {
    'clf__learning_rate': [0.03, 0.05, 0.08, 0.1, 0.15, 0.2],
    'clf__max_iter': [100, 200, 300, 500],
    'clf__max_leaf_nodes': [15, 31, 63, 127],
    'clf__min_samples_leaf': [20, 50, 100, 200],
    'clf__l2_regularization': [0.0, 0.1, 1.0, 10.0],
    'clf__max_features': [0.6, 0.8, 1.0],
}

search = RandomizedSearchCV(
    pipe, param_distributions, n_iter=30,
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
    scoring='balanced_accuracy', n_jobs=-1, random_state=42, verbose=1,
)
search.fit(X, y)
print(search.best_params_)
print(f"best 3-fold score: {search.best_score_:.5f}")

Fitting 3 folds for each of 30 candidates, totalling 90 fits
{'clf__min_samples_leaf': 20, 'clf__max_leaf_nodes': 31, 'clf__max_iter': 200, 'clf__max_features': 0.6, 'clf__learning_rate': 0.15, 'clf__l2_regularization': 1.0}
best 3-fold score: 0.94964


In [3]:
best_model = search.best_estimator_
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(best_model, X, y, cv=cv5, scoring='balanced_accuracy', n_jobs=-1)
print(scores.round(5))
print(f"5-fold mean: {scores.mean():.5f}  vs notebook 06 default: 0.94925")

[0.94973 0.95145 0.94889 0.94877 0.94795]
5-fold mean: 0.94936  vs notebook 06 default: 0.94925


In [4]:
best_model.fit(X, y)
preds = best_model.predict(X_test)
submission = pd.DataFrame({'id': test['id'], 'health_condition': preds})
submission.to_csv('../data/submission.csv', index=False)
print(submission['health_condition'].value_counts(normalize=True))

health_condition
at-risk      0.810622
unhealthy    0.115495
fit          0.073883
Name: proportion, dtype: float64


In [5]:
param_refined = {
    'clf__learning_rate': [0.08, 0.1, 0.12, 0.15, 0.18],
    'clf__max_iter': [200, 300, 400, 600],
    'clf__max_leaf_nodes': [31, 63],
    'clf__min_samples_leaf': [10, 20, 40],
    'clf__l2_regularization': [0.3, 1.0, 3.0, 10.0],
    'clf__max_features': [0.5, 0.6, 0.7],
}

search2 = RandomizedSearchCV(
    pipe, param_refined, n_iter=60,
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
    scoring='balanced_accuracy', n_jobs=-1, random_state=42, verbose=1,
)
search2.fit(X, y)
print(search2.best_params_)
print(f"best 3-fold: {search2.best_score_:.5f}")

Fitting 3 folds for each of 60 candidates, totalling 180 fits
{'clf__min_samples_leaf': 10, 'clf__max_leaf_nodes': 31, 'clf__max_iter': 300, 'clf__max_features': 0.6, 'clf__learning_rate': 0.12, 'clf__l2_regularization': 3.0}
best 3-fold: 0.94966


In [6]:
best_model2 = search2.best_estimator_
scores2 = cross_val_score(best_model2, X, y, cv=cv5, scoring='balanced_accuracy', n_jobs=-1)
print(scores2.round(5))
print(f"refined 5-fold: {scores2.mean():.5f}")
print(f"comparison  ->  default: 0.94925 | search1: 0.94936 | search2: {scores2.mean():.5f}")

[0.94995 0.95144 0.94898 0.94924 0.94831]
refined 5-fold: 0.94959
comparison  ->  default: 0.94925 | search1: 0.94936 | search2: 0.94959


In [7]:
best_model2.fit(X, y)
preds = best_model2.predict(X_test)
submission = pd.DataFrame({'id': test['id'], 'health_condition': preds})
submission.to_csv('../data/submission.csv', index=False)
print(submission['health_condition'].value_counts(normalize=True))

health_condition
at-risk      0.809980
unhealthy    0.116009
fit          0.074011
Name: proportion, dtype: float64
